In [ ]:
#           *** RETIRED CODE FROM DAAANALYZER_2.0 **
# I decided to just get data from analyzed and filtered fles output by ABR peak analysis. 
# 1. Open the folder and get all available ABR files
for daa in dataList:
    # Setup the path to the file
    abrFold = dirData+daa+'/'
    # Get a list of available ABR files
    allFiles = os.listdir(abrFold)
    files = []
    for file in allFiles:
        if ("ABR-"+daaRun) in file:
            files.append(file)
    # Go through the files and make a line plot that contains all ABRs for that frequency
    for file in files:
        # Setup the path to the file
        path = dirData+daa+'/'+file
        # Read in the file & make adjustments for readability
        dfABR = pd.read_csv(path, sep='\t', encoding='unicode_escape', lineterminator='\n')
        # Have to get out:
        freq = dfABR.columns.tolist()[6][8:-1]
        # Levels
        # Number of averages -- for each column showing 



str = dfABR.iloc[0].values[0][7:-1]
tries = [i.strip() for i in str.split(';')]
tries
# This one is going to be harder because the levels are embedded and it'll be hard to say how the indexing changes.
# Should pln on just searching for the word LEVELS: and then get all of the characters until the next "\r"
dfABR.iloc[1].values[0]

In [10]:
# *** READ IN EACH ISOLINE TSV FILE FOR EACH DPOAE**
# Iterate through the folders
dfAll = pd.DataFrame()
for daa in dataList:
    # Setup the path to the file
    path = dirData+daa+'/'+isoFile
    # Read in the file & make adjustments for readability
    dfIso = pd.read_csv(path,  sep='\t')
    dfIso['Vals'] = dfIso.index
    # Iterate through all of the rows available and get values 
    freqs = []
    threshs = []
    for index, row in dfIso.iterrows():
        f2Freq = row['Vals'][0]
        f2Freq = int(float(f2Freq)/1000)
        if (f2Freq != 64):
            freqs.append(f2Freq)
            thresh = row['Vals'][3]
            if thresh =='   NaN':
                thresh = 75
            threshs.append(int(float(thresh)))
    data = {'f2Freq_kHz': freqs, 'Thresh_dBSPL': threshs}
    df = pd.DataFrame(data)
    anID = daa[8:-2]
    df['AnimalID'] = anID
    df['CMIID'] = prepID
    df['DAA_Run'] = daaRun
    # Get information about this animal from the MD sheet
    inMD = dfMD.index[dfMD['AnimalID']==anID].tolist()[0]
    inj = dfMD.loc[inMD]['InjGroup']
    sex = dfMD.loc[inMD]['Sex']
    df['InjGroup'] = inj
    df['Sex'] = sex
    dfAll = pd.concat([dfAll,df])
dfAll.reset_index(inplace=True)
dfAll.to_csv(dirDFs+fnThs) 

In [8]:
#      *** GENERATE A LIST OF DAA FOLDERS THAT WILL BE ANALYZED **
# Get a list of all the folders that match the prep
allData = os.listdir(dirData)
dataList = []
for datum in allData:
    if ('.'+daaRun) in datum:
        dataList.append(datum)
dataList.sort()
dataList

['CMI_005.C032.2',
 'CMI_005.C033.2',
 'CMI_005.C034.2',
 'CMI_005.C035.2',
 'CMI_005.C036.2',
 'CMI_005.C037.2',
 'CMI_005.C038.2',
 'CMI_005.C039.2']

In [7]:
#    *** LOAD MD SHEETS **
# Sample metadata sheet that include animal and condition info
dfMD = pd.read_csv(dirMD+fnMD)
dfMD.reset_index(inplace=True)
dfMD

,index,AnimalID,Sex,DOB,CMIID,InjectionDate,AgeAtInjection,Package,InjGroup,NE,NEDate,AgeAtNE,NELevel,NEBand,NEDuration,NEGroup
0,0,C032,F,10/8/24,CMI_005,11/25/24,7,AAV,CtrlInj,NaN,NaN,NaN,97.5,8_16,2h,NaN
1,1,C033,M,10/8/24,CMI_005,11/25/24,7,AAV,CtrlInj,NaN,NaN,NaN,97.5,8_16,2h,NaN
2,2,C034,F,10/8/24,CMI_005,11/25/24,7,AAV,CtrlInj,NaN,NaN,NaN,97.5,8_16,2h,NaN
3,3,C035,M,10/8/24,CMI_005,11/25/24,7,AAV,HABclw,NaN,NaN,NaN,97.5,8_16,2h,NaN
4,4,C036,F,10/8/24,CMI_005,11/25/24,7,AAV,HABclw,NaN,NaN,NaN,97.5,8_16,2h,NaN
5,5,C037,M,10/8/24,CMI_005,11/25/24,7,AAV,HABclw,NaN,NaN,NaN,97.5,8_16,2h,NaN
6,6,C038,F,10/8/24,CMI_005,11/25/24,7,AAV,UnInj,NaN,NaN,NaN,97.5,8_16,2h,NaN
7,7,C039,M,10/8/24,CMI_005,11/25/24,7,AAV,UnInj,NaN,NaN,NaN,97.5,8_16,2h,NaN


In [ ]:
# *** IDENTIFY THRESHOLDS FOR EACH ABR FILE ***
#  DO NOT NEED TO DO THIS MORE THAN ONCE -- SEE CELL BELOW FOR CODE TO OPEN EXISTING FILE 
#       THAT ALREADY HAS BLINDED THRESHOLD DATA
#  Open the ABR threshold file made in the initialization notebook
dfThs = pd.read_csv(dirDFs+fnThs, index_col=0)
# Get list of all files to eval & randomize the order
allFiles = dfAll["AnimalFile"].unique()
random.shuffle(allFiles)
for file in allFiles:
    # Get metadata for this file
    inFile = dfAll.index[dfAll['AnimalFile']==file].tolist()[0]
    anID = dfAll.loc[inFile]['AnimalID']
    cmiID = dfAll.loc[inFile]['CMIID']
    sex = dfAll.loc[inFile]['Sex']
    injG = dfAll.loc[inFile]['InjGroup']
    fName = dfAll.loc[inFile]['File']
    daaRun = dfAll.loc[inFile]['DAA_Run']
    freq = dfAll.loc[inFile]['Freq_kHz']
    
    """ 
    # Setup graph to show user to get threshold for this file
    g = sns.FacetGrid(dfAll[dfAll['AnimalFile']==file], row="Level", aspect=4, height=1, 
                  row_order=np.sort(dfAll["Level"].unique())[::-1],hue="Level")
    g.map(sns.lineplot, "Time_ms", "Amp_uV")
    # Define and use a simple function to label the plot in axes coordinates
    def label(x, color, label):
        ax = plt.gca()
        ax.text(0, .5, label, fontweight="bold", color=color,
                ha="right", va="center", transform=ax.transAxes)
    g.map(label, "Level")
    # Adjust plot aesthetics
    g.set(xlim=(0, 9), ylim=(-2.5, 2.5))
    # Set the subplots to overlap
    g.figure.subplots_adjust(hspace=-.1)
    # Remove axes details that don't play well with overlap
    g.set_titles("")
    g.set(yticks=[], ylabel="")
    g.set(xlabel="Time (msec)")
    g.despine(bottom=True, left=True)
    plt.show()
    # Get threshold from user
    th = input("What is the threshold for this file?\n"+
                "Be sure to enter it as shown on the plot: ")
    clear_output(wait=True)
    """
    # Get threshold & 80 dB wave I amplitudes for this file
    #    Find the index for the th
    dfSS = dfAll[dfAll['AnimalFile']==file]
    inTh = dfSS.index[dfSS['Level']==th].tolist()[0]
    wOneTh = dfSS.loc[inTh]['P1_Amplitude_uV']
    inEdB = dfSS.index[dfSS['Level']=="80"].tolist()[0]
    wOneEdB = dfSS.loc[inEdB]['P1_Amplitude_uV']

    # Format information for this file into df
    data = {'AnimalID':anID,"CMIID":prepID,"Sex":sex,"InjGroup":injG, "File":fName,"DAA_Run":daaRun,
            "Freq_kHz":freq, "Th_dB":th, "W1uV_Th":wOneTh, "W1uV_80dB":wOneEdB}
    df = pd.DataFrame(data, index=[0])

    # Add info to df for all 
    dfTh = pd.concat([dfTh,df])

# Save the threshold df 
dfTh.reset_index(inplace=True)
#dfTh.drop(['level_0', 'Unnamed: 0','index'], axis=1, inplace=True)
dfTh.to_csv(dirDFs+fNThs) 

In [ ]:

                """
                # Read in the raw ABR file
                with open(abrFold+file, "r") as f:
                    rawABR = f.read()""
                
                    
                # Get the base filename
                strFile = str(file)
                inPost = strFile.index("analyzed.txt")
                basename = strFile[0:(inPost-1)]
                
                # Open the analyzed file to get the ABR frequency for this file
                with open(abrFold+basename+"-analyzed.txt", "r") as f:
                    anaABR = f.read()
                    inPre = anaABR.index("(kHz):")
                    inPost = anaABR.index("Threshold estimation:")
                    freq = anaABR[(inPre+6):(inPost-4)]
                # Read in the analyzed ABR file 
                dfAA = pd.read_csv(abrFold+basename+"-analyzed.txt", sep="\t", skiprows=6)
                # Read in the filtered ABR file 
                dfFA = pd.read_csv(abrFold+basename+"-filtered.txt", delimiter='\t', header=0,index_col="Time (ms)")
                # Get the number of dB levels for the file
                for level in dfFA.columns:
                    # Get the WI amplitudes for this level
                    inLV = dfAA.index[dfAA[dfAA.columns[0]]==float(level[0:-3])].tolist()[0]
                    waveOne = dfAA.loc[inLV]['P1 Amplitude']
                data = {'AnimalID':anID,"CMIID":prepID,"Sex":sex,"InjGroup":inj,
                        "AnimalFile":anID+"."+basename, "File":basename,"DAA_Run":daaRun,
                        "Freq_kHz":float(freq),"Level":float(level[0:-5]), 
                        "Time_ms":dfFA.index,"Amp_uV":dfFA[level].values,"P1_Amplitude_uV":waveOne}
                df = pd.DataFrame(data)
                df['AnimalID'] = anID
                df['CMIID'] = prepID
                df['DAA_Run'] = daaRun
                dfABRs = pd.concat([dfABRs,df])
                """

In [23]:
# **** MAKE GROUP SUMMARY ABR THRESHOLD PLOTS ****
# Get list of injGroups
maxY = 80.0
# Iterate through all frequencies and make one facet grid plot per freq
for freq in freqsToAna:
    dfSS = dfThs[dfThs["Freq_kHz"]==freq]
    dfSS['Threshold'] = dfThs['Th_dB_Blinded'].astype(float)
    ax = sns.pointplot(dfSS, x="Freq_kHz", y="Threshold",hue="InjGroup", palette=colors)
    ax.set_title(prepID+" | "+daaType+" | "+str(freq)+" kHz")
    ax.set(xlabel="Tone Frequency (kHz)")
    ax.set(ylabel="Threshold (dB SPL)")
    ax.set_ylim((0, maxY))

    plt.legend(title='Group', bbox_to_anchor=(1.05, 1), loc='upper left',  fancybox=True, labels=conditions)
    leg = ax.get_legend()
    noHandles = len(leg.legend_handles)
    i=0
    for i in range(0, noHandles):
        leg.legend_handles[i].set_color(legColors[i])
    plt.savefig(dirGpABRPlts+prepID+'.ABRThresholds.'+str(freq)+'kHz.DAA_'+daaRun+'.png',  bbox_inches='tight', dpi = 300,format = 'png')
    plt.close()